In [1]:
import polars as pl

In [2]:
books_df = pl.scan_ndjson("../processed-data/cleaned_books_fantasy_paranormal.json")
interactions_df = pl.scan_ndjson("../processed-data/cleaned_interactions_fantasy_paranormal.json")
authors_df = pl.scan_ndjson("../raw-data/goodreads_book_authors.json")
series_df = pl.scan_ndjson("../raw-data/goodreads_book_series.json")

In [3]:
# filter out descriptions with fewer than 50 words
books_df = books_df.filter(pl.col('description').str.count_matches(r"\w+") >= 50)

In [4]:
books_df.select('language_code').unique().show()

language_code
str
"""por"""
"""vie"""
"""aus"""
"""ita"""
"""pol"""


In [5]:
# Keep only english variations
# """en"""
# """en-CA"""
# """en-GB"""
# """en-US"""
# """eng"""

books_df = books_df.filter(
    pl.col('language_code').is_in(['en', 'en-CA', 'en-GB', 'en-US', 'eng'])
)


In [6]:
books_df = books_df.explode("authors").with_columns(
    pl.col("authors").struct.field("author_id").alias("author_id")
)
books_df = books_df.join(
    authors_df.select(["author_id", "name"]),
    on="author_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["authors", "author_id", "name"]).first(),
    pl.col("name").drop_nulls().alias("author_names")
)


In [7]:
books_df = books_df.explode("series").with_columns(
    pl.col("series").alias("series_id")
)
books_df = books_df.join(
    series_df.select(["series_id", pl.col("title").alias("series_title")]),
    on="series_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["series", "series_id", "series_title"]).first(),
    pl.col("series_title").drop_nulls().alias("series")
)


In [8]:
books_df = books_df.with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().struct.field("name")
    ).alias("popular_shelves")
).with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().filter(
            ~pl.element().str.contains(r"(?i)read|own|buy|fav|library|audio|kindle|ebook")
        )
    )
)


In [9]:
books_df.select('language_code').unique().show()

language_code
str
"""en"""
"""eng"""
"""en-US"""
"""en-GB"""
"""en-CA"""


In [10]:
books_df.head().show()

book_id,isbn,text_reviews_count,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,ratings_count,work_id,title,title_without_series,author_names,series
i64,str,i64,str,str,list[str],str,str,f64,str,list[str],str,str,str,str,i64,i64,str,i64,str,i64,str,str,i64,i64,str,str,list[str],list[str]
18108964,"""""",3,"""US""","""eng""","[""m-m"", ""paranormal"", … ""m-m-paranormal""]","""""","""true""",3.59,"""""",[],"""[Siren Everlasting Classic Man…","""ebook""","""https://www.goodreads.com/book…","""Siren-Bookstrand, Inc.""",99,29,"""9781627400718""",6,"""""",2013,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",65,25431841,"""The Alpha Wolf's Convenient Ma…","""The Alpha Wolf's Convenient Ma…","[""Marcy Jacks""]","[""The Vampire District""]"
9854729,"""""",40,"""US""","""en-GB""","[""romance"", ""sci-fi"", … ""how-about-no""]","""B004HO6626""","""true""",3.44,"""B004HO6626""","[""10801887"", ""6590607"", … ""988553""]","""The sequel to Space Junque. He…","""Kindle Edition""","""https://www.goodreads.com/book…","""""",187,2,"""""",1,"""""",2011,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",228,14745853,"""Spiderwork (Apocalypto, #2)""","""Spiderwork (Apocalypto, #2)""","[""L.K. Rigel""]","[""Apocalypto""]"
32478337,"""""",2,"""US""","""eng""","[""paranormal"", ""romance"", … ""novella-short-story""]","""""","""true""",4.01,"""""","[""8739676"", ""13055684"", … ""1418671""]","""Alternate cover image for  So…","""Kindle Edition""","""https://www.goodreads.com/book…","""""",null,12,"""""",5,"""""",2012,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",24,19446679,"""Kel (Zodius, #2.5)""","""Kel (Zodius, #2.5)""","[""Lisa Renee Jones""]","[""Zodius""]"
26953321,"""""",4,"""US""","""eng""","[""urban-fantasy"", ""fantasy"", … ""fantasy-scifi-wonderful-weird-crap""]","""""","""true""",4.27,"""""",[],"""The stage is set for Mr. Norse…","""ebook""","""https://www.goodreads.com/book…","""Serial Box Publishing""",null,null,"""9781682100189""",null,"""""",2015,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",45,47003696,"""Codex Umbra (Bookburners #1.11…","""Codex Umbra (Bookburners #1.11…","[""Mur Lafferty"", ""Margaret Dunlap"", … ""Max Gladstone""]","[""Bookburners""]"
34941586,"""""",16,"""US""","""eng""","[""fantasy"", ""fiction"", … ""contemporary-fantasy""]","""""","""true""",4.11,"""""","[""186427"", ""6931246"", … ""77207""]","""This is an alternate cover edi…","""Kindle Edition""","""https://www.goodreads.com/book…","""William Morrow""",674,null,"""""",4,"""Tenth Anniversary Edition""",2017,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",52,1970226,"""American Gods (American Gods, …","""American Gods (American Gods, …","[""Neil Gaiman""]","[""American Gods""]"


In [11]:
books_df = books_df.drop([
    'isbn', 'text_reviews_count', 'country_code', 'language_code',
    'is_ebook', 'kindle_asin', 'format', 'num_pages',
    'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'image_url',
    'title_without_series', 'ratings_count', 'publisher', 'similar_books', 'asin', 'work_id'
])
books_df.head().show()


book_id,popular_shelves,average_rating,description,link,url,title,author_names,series
i64,list[str],f64,str,str,str,str,list[str],list[str]
8524790,"[""fantasy"", ""young-adult"", … ""signed-books""]",3.8,"""When Wayland North brings rain…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""Brightly Woven""","[""Alexandra Bracken""]",[]
28580164,"[""fantasy"", ""magazines"", … ""kickstarter""]",3.45,"""The January/February 2016 issu…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""Uncanny Magazine Issue 8: Janu…","[""Sarah Rees Brennan"", ""Rose Lemberg"", … ""Maria Dahvana Headley""]","[""Uncanny Magazine""]"
6569734,"[""young-adult"", ""paranormal"", … ""vampires""]",4.07,"""Ghosts ruin everything. Especi…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""Ninth Key (The Mediator, #2)""","[""Jenny Carroll"", ""Meg Cabot""]","[""The Mediator""]"
465734,"[""fantasy"", ""childrens"", … ""fantasy-childrens""]",3.86,"""FERGUS CRANE! YOU ARE IN GREAT…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""Fergus Crane""","[""Paul Stewart""]","[""Far-Flung Adventures""]"
1154166,"[""fiction"", ""young-adult"", … ""dnf""]",3.99,"""The Barnes & Noble Review Ursu…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""A Wizard of Earthsea (The Eart…","[""Ursula K. Le Guin"", ""Harlan Ellison""]","[""Earthsea Cycle""]"


In [12]:
books_df = books_df.with_columns(
    pl.concat_str(
        [
            pl.col("title").fill_null(""),
            pl.col("title").fill_null(""),
            pl.col("title").fill_null(""),
            pl.col("author_names").list.join(", ").fill_null(""),
            pl.col("author_names").list.join(", ").fill_null(""),
            pl.col("series").list.join(", ").fill_null(""),
            pl.col("popular_shelves").list.join(", ").fill_null(""),
            pl.col("description").fill_null("")
        ],
        separator=" "
    ).alias("combined_text")
)


In [13]:
# Basic text preprocessing on combined_text
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS = set(stopwords.words('english'))
stopwords_regex = r"(?i)\b(" + "|".join(STOPWORDS) + r")\b"

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    if not text:
        return ""
    return " ".join([lemmatizer.lemmatize(w) for w in text.split()])

books_df = books_df.with_columns(
    pl.col("combined_text")
    .str.to_lowercase()
    .str.replace_all(r"[^\w\s]", " ") # remove punctuation
    .str.replace_all(stopwords_regex, "") # remove stopwords
    .str.replace_all(r"\s+", " ") # remove multiple spaces
    .str.strip_chars() # trim leading/trailing spaces
    .map_elements(lemmatize_text, return_dtype=pl.String)
)

books_df.head().show()


book_id,popular_shelves,average_rating,description,link,url,title,author_names,series,combined_text
i64,list[str],f64,str,str,str,str,list[str],list[str],str
22343904,"[""graphic-novels"", ""fantasy"", … ""fantasy-epic""]",4.51,"""Over thirty-five years after i…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""The Complete ElfQuest, Volume …","[""Wendy Pini"", ""Richard Pini""]","[""The Complete ElfQuest""]","""complete elfquest volume two c…"
12046262,"[""romance"", ""paranormal"", … ""blog-tour""]",4.39,"""Marie Kincaid is devastated af…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""Glazier""","[""Bri Clark""]",[],"""glazier glazier glazier bri cl…"
10585894,"[""steampunk"", ""fantasy"", … ""mystery""]",3.86,"""Book One of the Chronicles of …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""Reaper's Return (Chronicles of…","[""Ren Cummins""]","[""Chronicles of Aesirium""]","""reaper return chronicle aesiri…"
22431722,"[""another-world-fantasy"", ""magic"", … ""paranormal-series""]",4.67,"""Regan's arrival in Ireland may…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""Gift of Prophecy (Gifted #3)""","[""Erin Manbeck""]","[""Gifted""]","""gift prophecy gifted 3 gift pr…"
17164661,"[""science-fiction"", ""young-adult"", … ""when-the-sil-enc-e-ends""]",4.26,"""When you choose your friends, …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…","""When the Silence Ends""","[""Jade Kerrion""]","[""Double Helix Case Files""]","""silence end silence end silenc…"


In [14]:
import os
os.makedirs("../processed-data", exist_ok=True)
books_df.collect().write_ndjson("../processed-data/processed_books_texts.json")
print("Dataset successfully saved!")

Dataset successfully saved!
